# Pseudo-labeling test — menggeser komposisi training ke distribusi test

Satu-satunya sumbu yang belum pernah dijalankan setelah tiga batch Kaggle. `Net3` identik notebook_8; yang berubah cuma **himpunan record yang dilatih**.

**Waktu: ~85 menit (arm A) + ~170 menit (arm B) = ~4,3 jam di T4.** Arm B dua kali lipat karena tiap fold melatih dua model bersilang atas belahan test — itu yang menutup kebocoran label-semu ke prediksi test.

## Aritmatika target — baca ini dulu

Kalibrasi dari 3 submission nyata (R2 0,92): **LB ≈ `lb_real` + 0,0082** (sd 0,0028).

| Target LB | butuh `lb_real` | Δ dari nb8 (0,1762) |
|---|---|---|
| 0,1837 (peringkat 1) | 0,1755 | −0,0007 |
| 0,1800 | 0,1718 | −0,0044 |
| 0,1750 | 0,1668 | −0,0094 |
| **0,1700** | **0,1618** | **−0,0144** |
| **0,1600** | **0,1518** | −0,0244 |

Terobosan terbesar yang pernah ditemukan di repo ini — CNN v3 di `notebook_8` — bernilai
**−0,0070**. Jadi **LB 0,17 butuh dua kali lipat terobosan terbesar itu**.

**LB 0,16 butuh `lb_real` 0,1518, dan itu di BAWAH batas oracle-grouping 0,1543** — yaitu
skor yang dicapai kalau kita *tahu persis* record mana milik event mana. Post-processing
tidak bisa menembusnya dengan model dasar sekarang; hanya model per-record yang jauh lebih
akurat yang bisa. Perlakukan **0,16 sebagai di luar jangkauan** pendekatan ini.

Karena itu tiap notebook mencetak verdikt eksplisit terhadap kelima ambang di atas.


## Cara pakai

1. **Settings -> Accelerator -> GPU** (T4 atau P100).
2. Tambahkan kompetisinya lewat **Add Input**.
3. Upload folder `belum_run/ref/` sebagai **Kaggle Dataset**, lalu Add Input juga.
   Isinya `nb8_ref.npz` — prediksi 8 kandidat notebook_8. Tanpa file itu notebook tetap
   jalan, tapi hanya bisa melaporkan skor model tunggal dan **tidak bisa memberi verdikt
   terhadap target LB**.
4. Run All.
5. Sel terakhir membungkus semua hasil jadi **satu file zip** di `/kaggle/working/`.
   Unduh zip itu dari panel Output, ekstrak `.npz`-nya ke `Case 1/cache/`.

## Yang perlu dibaca sebelum percaya angka apapun

`eb_twmae` mentah **cacat** di sisi depth-bulat: rekonstruksi event lewat
`groupby(depth_km, magnitude)` melebur record depth-bulat jadi pseudo-event sampai
**401 record**, sehingga CV over-weight ekor sekitar **3x**. Notebook ini memakai
`lb_real`, bukan `eb_twmae`.

Dan `lb_real` sendiri mengukur **data train**. `bias_rnd` serta mean dan sd prediksi
subset depth-bulat mengukur **jarak ke test**. Kalau keduanya bertabrakan, **yang kedua
menang** — `notebook_2` dan `notebook_5` dua-duanya memperbaiki CV lalu memburuk di LB.

---

## Kenapa pseudo-labeling, setelah tiga batch gagal

Sumbu yang sudah tertutup dengan bukti:

| sumbu | vonis |
|---|---|
| arsitektur CNN 1D baru | 7 dari 8 notebook gagal, korelasi 0,89–0,97 vs CNN v3 |
| grouping / co-membership | 2 usaha independen, panen 0,3–1,0% headroom |
| post-processing / kalibrasi / koreksi ekor | menang 1/12; koreksi ekor terbukti terbalik arah |
| 196 fitur masuk ke dalam CNN | nb10: `bias_rnd` +0,115, menang 0/12 |
| TTA + bobot record clipped | UJI 2: gain +0,00029, menang 6/12 |
| menggeser bobot blend nb8 | nb9 dan nb10 dua-duanya rugi di LB |

**Pseudo-labeling tidak ada di daftar itu.** `experiments/exp34_pseudo.py` berhenti setelah
menghitung labelnya; trainingnya tidak pernah dijalankan.

### Mekanismenya, dalam satu angka

```
proporsi record depth-bulat:   32% di TRAIN   vs   51% di TEST
```

Semua model di repo ini dilatih pada komposisi yang salah, dan sisi depth-bulat adalah
tempat **seluruh** headroom yang tersisa (dec 0,1437 sudah melewati oracle-nya; rnd 0,2074).
Pseudo-label menambah 2.300 record dari distribusi yang benar — 22% tambahan, dan
proporsinya condong persis ke sisi yang kurang terwakili.

### Kenapa dua kegagalan yang mirip TIDAK memvonis ini

**Density-ratio nb7 (gagal, `lb_real` 0,2245 vs 0,2101).** Itu memberi bobot lebih besar ke
record train yang sudah ada. Reweighting tidak bisa menciptakan record depth-bulat yang
tidak ada di train; ia cuma memperbesar suara record yang sama. PL menambah record baru.

**Pra-latih SSL nb07 (gagal, 0,2112 vs 0,2074).** Itu memakai waveform test **tanpa label**
— hipotesis "kurang data mentah" dan hipotesis itu memang mati. PL menambah **sinyal
supervisi** pada komposisi target, bukan data mentah. Pertanyaannya berbeda.

## Jebakan senyap yang ditutup notebook ini

Kalau record test dilatih dengan pseudo-label dari blend nb8, model mencocokkan record itu,
jadi **prediksi test-nya runtuh ke arah nb8** — sementara OOF (train) tetap terlihat
beragam. Pencarian bobot membaca diversitas dari OOF lalu menerapkannya ke prediksi test
yang sudah kolaps. **Tidak satu pun diagnostik yang ada sekarang bisa menangkap itu.**

Penutupnya: test dibelah dua menurut `depth_km` (record satu event tidak terpisah), dan
tiap fold melatih dua model bersilang —

```
model 1: train[ti] + test[belahan 0 berlabel-semu]  ->  memprediksi test[belahan 1]
model 2: train[ti] + test[belahan 1 berlabel-semu]  ->  memprediksi test[belahan 0]
```

Tidak ada record test yang diprediksi oleh model yang pernah melihat label-semunya sendiri.
Ongkosnya 2x training, dan itu memang dibayar.

**Diagnostik penentu untuk jebakan ini** (dicetak di sel 5): korelasi `pte` kandidat baru
terhadap `pte` blend nb8. Kalau ia melonjak jauh di atas korelasi `oof`-nya, kolapsnya
terjadi dan verdikt blend tidak sah — berapa pun bagus `lb_real`-nya.

## Arm A adalah KONTROL, dan itu wajib

Resepnya identik `cnn3`, jadi `lb_real` di K=1 harus mendarat di **0,1898**
(angka **SETELAH** `post_tr`; `lb_real` mentahnya 0,2091 — jangan tertukar, itu yang
merusak verdikt notebook 09).

Arm A menjawab dua hal sekaligus:

1. **Pembanding se-lingkungan.** Verdikt blend nb09 mengadu kandidat Kaggle melawan
   8 kandidat lokal, dan gap lingkungan 0,0049 masuk sebagai kerugian ide. Dengan arm A
   di slot CNN, kandidat dan pembanding sama-sama dari Kaggle.
2. **Gap lingkungan itu sendiri**, yang sampai sekarang belum dijelaskan: korelasi OOF
   resep identik cuma 0,9722, padahal antar-paket 3-seed lokal 0,9875. Sel Setup sekarang
   mencetak **sidik jari input** (jumlah |.| Wtr/Str/Wte/Ste/A_BASE) terhadap acuan lokal.
   Kalau cocok, inputnya identik dan selisihnya lahir di training; kalau tidak, sebaliknya.

## Yang harus diabaikan di notebook ini

Baris **"gain sisi desimal"** dicetak sebagai informasi, **bukan kriteria seleksi**. Aturan
lama "gain sisi desimal harus negatif" sudah DICABUT — `notebook_10` dibangun untuk
memaksimalkannya dan LB-nya rugi 0,0022.

## Kalibrasi sebelum bersemangat

LB publik cuma 30% data test; sd selisih dua submission berbeda **0,0038**. LB terbaik
0,1849, peringkat 1 0,1835 — selisih 0,0014 = **0,4 sigma**. Target LB 0,1800 menuntut
`lb_real` **0,1714**, yaitu **−0,0048** dari 0,1762. Belum ada satu ide pun setelah nb8
yang tembus segitu. Dan **hanya ada satu submission final** — tidak ada lindung nilai.


---
## 0. Setup, metrik, fitur

Semua kode inti disalin apa adanya dari `lab/` supaya angkanya sebanding dengan run lokal.

In [ ]:
import os, sys, time, warnings, json, zipfile, glob
from pathlib import Path
import numpy as np, pandas as pd
from scipy import signal, stats
warnings.filterwarnings("ignore")

KG = "/kaggle/input/competitions/ai-innovation-case-task-1-seismic-magnitude-prediction/seismic-dataset"
DATA = Path(KG) if os.path.exists(KG) else Path("seismic-dataset")
OUT  = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path("cache")
OUT.mkdir(exist_ok=True, parents=True)

# nb8_ref.npz = prediksi 8 kandidat notebook_8 (OOF + test) + bobot blend-nya.
# Diperlukan untuk memberi VERDIKT terhadap target LB. Upload folder `belum_run/ref/`
# sebagai Kaggle Dataset, atau taruh nb8_ref.npz di mana saja di bawah /kaggle/input.
# Lokasi yang sudah diketahui lebih dulu, baru penelusuran rekursif sebagai cadangan
# (kalau slug dataset-nya berubah, atau dijalankan di lokal).
REF_KNOWN = ["/kaggle/input/datasets/timotiuspatrick/nb8-ref/nb8_ref.npz",
             "/kaggle/input/nb8-ref/nb8_ref.npz"]
def _find_ref():
    for p in REF_KNOWN:
        if os.path.exists(p): return Path(p)
    for root in ("/kaggle/input", "/kaggle/working", ".", "..",
                 "kaggle/ref", "../kaggle/ref"):
        if not os.path.isdir(root): continue
        for dp, _, fs in os.walk(root):
            if "nb8_ref.npz" in fs: return Path(dp)/"nb8_ref.npz"
    return None
REFP = _find_ref()
print("DATA :", DATA, "| ada:", DATA.exists())
print("REF  :", REFP)
if REFP is None:
    print()
    print("!"*72)
    print("nb8_ref.npz TIDAK DITEMUKAN -- notebook 01 akan berhenti di sel Training.")
    print("!"*72)
    print("File ini (334 KB) berisi prediksi OOF + test 8 kandidat notebook_8 beserta")
    print("bobot blend-nya. Tanpa itu tidak ada acuan untuk dibandingkan, dan notebook 01")
    print("tidak punya apa-apa untuk dikoreksi.")
    print()
    print("Cara memasangnya:")
    print("  1. Di repo lokal, ambil  Case 1/belum_run/ref/nb8_ref.npz")
    print("  2. Kaggle -> Datasets -> New Dataset -> upload file itu -> Create")
    print("  3. Balik ke notebook ini -> panel kanan -> Input -> + Add Input")
    print("     -> tab Datasets -> cari dataset barusan -> tanda +")
    print("  4. Run All lagi. Baris REF di atas harus berubah jadi sebuah path.")
    print()
    print("Isi /kaggle/input sekarang:")
    if os.path.isdir("/kaggle/input"):
        for d in sorted(os.listdir("/kaggle/input")): print("   -", d)
    else:
        print("   (/kaggle/input tidak ada -- ini bukan lingkungan Kaggle)")
    print("!"*72)
import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("DEV  :", DEV, torch.cuda.get_device_name(0) if DEV=="cuda" else "")
if DEV != "cuda":
    print("PERINGATAN: GPU tidak aktif. Settings -> Accelerator -> GPU.")


In [ ]:
FS, P_IDX, EPS = 200, 200, 1e-15
train = (pd.read_csv(DATA/"train"/"train_meta.csv")
           .merge(pd.read_csv(DATA/"train"/"train_labels.csv"), on="sample_id"))
test  = pd.read_csv(DATA/"test"/"test_meta.csv")
y   = train.magnitude.values.astype(np.float64)
# Tidak ada event_id di dataset, padahal metriknya per-event. Rekonstruksi heuristik:
ev  = train.groupby(["depth_km","magnitude"]).ngroup().values
cnt = pd.Series(ev).map(pd.Series(ev).value_counts()).values

def mag_weight(m):
    m=np.asarray(m); return np.where(m<5.5,1.0,np.where(m<6.0,1.5,np.where(m<6.5,2.0,2.5)))
def eb_twmae(yt,yp,e):
    df=pd.DataFrame({"y":yt,"p":yp,"e":e}); df["err"]=np.abs(df.y-df.p)
    g=df.groupby("e").agg(mae=("err","mean"),m=("y","first"))
    w=mag_weight(g.m.values); return float((w*g.mae.values).sum()/w.sum())

SW   = mag_weight(y)/cnt
d_tr = train.depth_km.values; d_te = test.depth_km.values
rnd_tr = np.isclose(d_tr, np.round(d_tr)); rnd_te = np.isclose(d_te, np.round(d_te))
MR = rnd_tr & (d_tr < 100); MT = rnd_te & (d_te < 100)
FD = 0.49   # proporsi event depth-desimal di test

# ---- METRIK lb_real -------------------------------------------------------------
# eb_twmae mentah CACAT di sisi depth-bulat: groupby(depth_km, magnitude) melebur record
# depth-bulat jadi pseudo-event sampai 401 record, sehingga CV over-weight ekor ~3x.
# lb_real memecah pseudo-event depth-bulat DANGKAL jadi chunk seukuran event nyata.
# JANGAN pernah menilai kandidat dengan eb_twmae mentah.
def make_ev_real(seed, chunk=8):
    rng=np.random.RandomState(seed); e2=ev.copy(); nxt=ev.max()+1
    split_me = rnd_tr & (d_tr < 100)     # >=100 km: grup 67-100% murni, jangan dipecah
    for eid in np.unique(ev[split_me]):
        idx=np.where(ev==eid)[0]
        if len(idx) <= chunk*1.5: continue
        rng.shuffle(idx); k=max(2,int(round(len(idx)/chunk)))
        for j,part in enumerate(np.array_split(idx,k)):
            if j>0: e2[part]=nxt; nxt+=1
    return e2
EVR = [make_ev_real(s) for s in range(8)]
def part_real(p, dec):
    m=(~rnd_tr) if dec else rnd_tr
    return float(np.mean([eb_twmae(y[m],p[m],e[m]) for e in EVR]))
def lb_real(p):  return FD*part_real(p,True)+(1-FD)*part_real(p,False)
def bias_rnd(p): return float(p[MR].mean()-y[MR].mean())
def make_sw(cap=None, power=1.0):
    c=cnt.astype(float)
    if cap is not None: c=np.minimum(c,cap)
    return mag_weight(y)/(c**power)

# KALIBRASI lb_real -> LB, dari 3 titik submission nyata (R2 0,92):
#   nb3 0,1945 -> 0,1996 | nb6 0,1845 -> 0,1952 | nb8 0,1762 -> 0,1849
# Kalibrasi lb_real -> LB, 5 titik:
#   nb3 +0,0051 | nb6 +0,0107 | nb8 +0,0087 | nb9 +0,0138 (OUTLIER) | nb10 +0,0098
# Tanpa nb9: mean +0,0086, sd 0,0025 -- cukup rapat. nb9 satu-satunya kegagalan ARAH,
# dan sebabnya spesifik (set-transformer menyeragamkan prediksi dalam grup depth).
LB_OFFSET = 0.0086          # mean 4 titik tanpa outlier nb9, sd 0,0025
LB_JUARA  = 0.1762          # lb_real notebook_8 (LB 0,1849). Kandidat harus MENGALAHKAN ini.
def lb_hat(v): return v + LB_OFFSET
print("event efektif:", len(np.unique(ev)), "| record:", len(y))


In [ ]:
class Scorer:
    """eb_twmae versi bincount atas banyak realisasi event -- ~5000x lebih cepat."""
    def __init__(self, evs, keep):
        self.parts=[]
        for dec,f in ((True,FD),(False,1-FD)):
            m = keep & ((~rnd_tr) if dec else rnd_tr)
            idx=np.where(m)[0]; per=[]
            for e in evs:
                _,inv=np.unique(e[idx],return_inverse=True)
                nrec=np.bincount(inv).astype(float)
                mm=np.zeros(len(nrec)); mm[inv]=y[idx]
                w=mag_weight(mm); per.append((inv,nrec,w,w.sum(),len(nrec)))
            self.parts.append((idx,f,per))
        self.n=len(evs)
    def __call__(self, p):
        tot=0.0
        for idx,f,per in self.parts:
            err=np.abs(y[idx]-p[idx]); s=0.0
            for inv,nrec,w,wsum,ne in per:
                s+=(w*(np.bincount(inv,weights=err,minlength=ne)/nrec)).sum()/wsum
            tot+=f*s/self.n
        return tot

class Post:
    """smoothing per grup depth + ekspansi varians PER SUBSET.

    Ekspansi harus dipusatkan per subset: versi global mendorong subset depth-bulat
    (yang rata-ratanya di atas mean keseluruhan) ikut naik dan memperbesar bias_rnd.
    Memusatkan per subset memangkas bias_rnd +0,076 -> +0,047 dengan ongkos CV 0,003."""
    def __init__(self, dep, lam_dec=0.95, lam_deep=0.5, T_deep=150, a_dec=1.15, a_rnd=1.05):
        r=np.isclose(dep,np.round(dep))
        self.lam=np.where(~r, lam_dec, np.where(dep>=T_deep, lam_deep, 0.0))
        order=np.argsort(dep,kind="stable"); self.order=order
        ds=dep[order]; bnd=np.flatnonzero(np.diff(ds))+1
        self.slices=np.concatenate([[0],bnd,[len(ds)]])
        self.r=r; self.a_dec=a_dec; self.a_rnd=a_rnd
    def __call__(self, p):
        ps=p[self.order]; gm=np.empty_like(ps); sl=self.slices
        for i in range(len(sl)-1): gm[sl[i]:sl[i+1]]=np.median(ps[sl[i]:sl[i+1]])
        g=np.empty_like(p); g[self.order]=gm
        q=(1-self.lam)*p+self.lam*g; out=q.copy()
        for m,a in ((self.r,self.a_rnd),(~self.r,self.a_dec)):
            mu=q[m].mean(); out[m]=mu+a*(q[m]-mu)
        return out

class SideScorer:
    """eb_twmae di SATU sisi saja. Dipakai untuk DIAGNOSTIK KE-4 (uji sisi-desimal).

    notebook_9 lulus ketiga diagnostik transfer lama, `lb_real`-nya membaik -0,0044, ia
    menang 12/12 belahan -- dan LB-nya tetap MEMBURUK (0,1849 -> 0,1856). Sebabnya:
    `lb_real` memperbaiki BOBOT metrik tapi bukan LABEL-nya. Record dari gempa berbeda
    yang berbagi (depth, magnitude) tetap punya `y` identik, jadi metode apa pun yang
    menyeragamkan prediksi di dalam satu grup depth dibayar berlebihan di sisi bulat.

    Sisi desimal tidak punya masalah itu (event-nya nyata, 98% murni). Jadi: perubahan
    yang menang di sisi bulat tapi RUGI di sisi desimal adalah bendera merah."""
    def __init__(self, dec, keep):
        m = keep & ((~rnd_tr) if dec else rnd_tr)
        self.idx=np.where(m)[0]; self.per=[]
        for e in EVR:
            _,inv=np.unique(e[self.idx],return_inverse=True)
            nrec=np.bincount(inv).astype(float)
            mm=np.zeros(len(nrec)); mm[inv]=y[self.idx]
            w=mag_weight(mm); self.per.append((inv,nrec,w,w.sum(),len(nrec)))
    def __call__(self, p):
        err=np.abs(y[self.idx]-p[self.idx]); s=0.0
        for inv,nrec,w,wsum,ne in self.per:
            s+=(w*(np.bincount(inv,weights=err,minlength=ne)/nrec)).sum()/wsum
        return s/len(self.per)

sc      = Scorer(EVR, np.ones(len(y), bool))
post_tr = Post(d_tr); post_te = Post(d_te)


In [ ]:
def _hp(a,f=0.075,order=2):
    b,aa=signal.butter(order,f/(FS/2),btype="high")
    return signal.filtfilt(b,aa,a,axis=1).astype(np.float32)
def L(x): return np.log10(np.abs(x)+EPS).astype(np.float32)
WINS=[0.25,0.5,1.0,1.5,2.0,3.0,4.0]

def extract(X, meta):
    t0=time.time(); out={}
    A=X.astype(np.float32); V=np.cumsum(_hp(A),axis=1)/FS; D=np.cumsum(_hp(V),axis=1)/FS
    def hv(W): return np.sqrt(W[:,:,0]**2+W[:,:,1]**2), np.abs(W[:,:,2])
    for nm,W in [("a",A),("v",V),("d",D)]:
        H,Vt=hv(W); allc=np.abs(W).max(axis=2)
        for s in WINS:
            n=int(s*FS)
            out[f"P{nm}_{s:g}"]=L(allc[:,P_IDX:P_IDX+n].max(1))
            out[f"P{nm}H_{s:g}"]=L(H[:,P_IDX:P_IDX+n].max(1))
            out[f"P{nm}V_{s:g}"]=L(Vt[:,P_IDX:P_IDX+n].max(1))
        out[f"pre_{nm}"]=L(allc[:,:P_IDX].max(1))
        out[f"snr_{nm}"]=out[f"P{nm}_4"]-out[f"pre_{nm}"]
        for s in [1.0,2.0,4.0]:
            n=int(s*FS)
            out[f"rms{nm}_{s:g}"]=L(np.sqrt((W[:,P_IDX:P_IDX+n,:]**2).mean(axis=(1,2))))
        out[f"hv_{nm}"]=L(H[:,P_IDX:].max(1))-L(Vt[:,P_IDX:].max(1))
    e=(A[:,P_IDX:,:]**2).sum(axis=2); ce=np.cumsum(e,axis=1); tot=ce[:,-1:]+EPS
    frac=ce/tot; out["logE"]=L(tot[:,0])
    for q in [0.1,0.25,0.5,0.75,0.9]:
        out[f"tE{int(q*100)}"]=(np.argmax(frac>=q,axis=1)/FS).astype(np.float32)
    out["E_slope"]=(out["tE90"]-out["tE10"]).astype(np.float32)
    for nm,W in [("a",A),("d",D)]:
        ab=np.abs(W[:,P_IDX:,:]).max(axis=2); tp=ab.argmax(axis=1)/FS
        out[f"tpk_{nm}"]=tp.astype(np.float32)
        out[f"rise_{nm}"]=(L(ab.max(1))/(tp+0.05)).astype(np.float32)
    n3=600
    num=(V[:,P_IDX:P_IDX+n3,:]**2).sum(1); den=(D[:,P_IDX:P_IDX+n3,:]**2).sum(1)
    out["tauc"]=L(2*np.pi*np.sqrt((den/(num+1e-30)).mean(1)))
    for s in [1.0,2.0]:
        n=int(s*FS); nu=(V[:,P_IDX:P_IDX+n,:]**2).sum(1)
        dv=np.gradient(V[:,P_IDX:P_IDX+n,:],1/FS,axis=1)
        out[f"taup_{s:g}"]=L(2*np.pi*np.sqrt((nu/((dv**2).sum(1)+1e-30)).mean(1)))
    for nm,W in [("a",A),("d",D)]:
        f,Pxx=signal.welch(W[:,P_IDX:,:],fs=FS,nperseg=256,axis=1)
        Pm=Pxx.mean(axis=2); tote=Pm.sum(1)+EPS
        for lo,hi in [(0.1,1),(1,3),(3,5),(5,10),(10,20),(20,50),(50,100)]:
            m=(f>=lo)&(f<hi)
            out[f"bd{nm}_{lo:g}_{hi:g}"]=L(Pm[:,m].sum(1))
            out[f"bdr{nm}_{lo:g}_{hi:g}"]=(Pm[:,m].sum(1)/tote).astype(np.float32)
        out[f"cent_{nm}"]=((Pm*f).sum(1)/tote).astype(np.float32)
        out[f"fdom_{nm}"]=f[Pm.argmax(1)].astype(np.float32)
        out[f"lhr_{nm}"]=L(Pm[:,(f>=0.1)&(f<3)].sum(1))-L(Pm[:,(f>=10)&(f<50)].sum(1))
    seg=A[:,P_IDX:,:]
    out["kurt"]=stats.kurtosis(seg,axis=1).mean(1).astype(np.float32)
    out["skew"]=np.abs(stats.skew(seg,axis=1)).mean(1).astype(np.float32)
    out["zcr"]=(np.diff(np.sign(seg),axis=1)!=0).mean(axis=(1,2)).astype(np.float32)
    Sg=seg-seg.mean(1,keepdims=True)
    evl=np.linalg.eigvalsh(np.einsum("nti,ntj->nij",Sg,Sg)/Sg.shape[1])
    evl=np.maximum(evl,0)+1e-30
    out["rectilin"]=(1-(evl[:,0]+evl[:,1])/(2*evl[:,2])).astype(np.float32)
    out["planarity"]=(1-2*evl[:,0]/(evl[:,1]+evl[:,2])).astype(np.float32)
    F=pd.DataFrame(out)
    dist=meta.distance_km.values.astype(np.float32); dep=meta.depth_km.values.astype(np.float32)
    F["distance_km"]=dist; F["depth_km"]=dep
    F["snr_db"]=meta.snr_db.values.astype(np.float32)
    F["is_clipped"]=meta.is_clipped.values.astype(np.int8)
    F["logdist"]=np.log10(dist+1); F["logdep"]=np.log10(dep+1)
    F["hypo"]=np.sqrt(dist**2+dep**2); F["loghypo"]=np.log10(F.hypo+1)
    for nm in ["a","v","d"]:
        for s in WINS:
            F[f"C{nm}_{s:g}"]=F[f"P{nm}_{s:g}"]+F["loghypo"]
            F[f"C{nm}d_{s:g}"]=F[f"P{nm}_{s:g}"]+1.5*F["logdist"]
        for s1,s2 in [(0.5,1.0),(1.0,2.0),(2.0,4.0),(0.5,4.0)]:
            F[f"g{nm}_{s1:g}_{s2:g}"]=F[f"P{nm}_{s2:g}"]-F[f"P{nm}_{s1:g}"]
    print("  extract %.0fs -> %d fitur"%(time.time()-t0,F.shape[1]))
    return F.astype(np.float32)

def build_input(X, meta):
    """Input CNN gaya MagNet: BENTUK gelombang dinormalisasi per-record, lalu skala
    amplitudo disuntikkan balik sebagai skalar.

    Catatan lama 'jangan normalisasi amplitudo per-record' benar untuk fitur pohon dan
    SALAH TOTAL untuk CNN -- waveform mentah membentang 5 orde magnitudo dan konvolusi
    tidak bisa belajar apapun dari itu. Aturan tanpa cakupan itu menahan model deep
    selama empat notebook."""
    A=X.astype(np.float32); V=np.cumsum(_hp(A),axis=1)/FS; D=np.cumsum(_hp(V),axis=1)/FS
    chans=[]; scales=[]
    for W in (A,V,D):
        mx=np.abs(W).max(axis=(1,2),keepdims=True)+1e-20
        chans.append((W/mx).transpose(0,2,1)); scales.append(np.log10(mx[:,0,0]))
    Wn=np.concatenate(chans,axis=1).astype(np.float32)
    dist=meta.distance_km.values.astype(np.float32); dep=meta.depth_km.values.astype(np.float32)
    S=np.stack([*scales, np.log10(dist+1), np.log10(dep+1),
                np.log10(np.sqrt(dist**2+dep**2)+1),
                meta.snr_db.values.astype(np.float32)/40.0,
                meta.is_clipped.values.astype(np.float32)],axis=1).astype(np.float32)
    return Wn, S

Xtr_raw = np.load(DATA/"train"/"train_waveforms.npy")
Xte_raw = np.load(DATA/"test"/"test_waveforms.npy")
t0=time.time()
Ftr = extract(Xtr_raw, train); Fte = extract(Xte_raw, test)
Wtr, Str = build_input(Xtr_raw, train)
Wte, Ste = build_input(Xte_raw, test)
del Xtr_raw, Xte_raw
print("siap %.0fs | Wtr %s  Str %s"%(time.time()-t0, Wtr.shape, Str.shape))

# ---------------------------------------------------------------------------------
# SIDIK JARI INPUT -- memisahkan "beda data" dari "beda training" kalau hasilnya meleset.
#
# Notebook 09 menghasilkan model 0,0049 lebih buruk dari run lokal untuk resep yang
# IDENTIK, dengan korelasi OOF cuma 0,9722 (antar-paket 3-seed lokal: 0,9875). Kode
# `extract()` dan `build_input()` sudah dibandingkan baris demi baris dan sama persis,
# jadi tersangkanya versi library -- tapi itu tidak pernah bisa dibuktikan karena
# tidak ada satu pun angka input yang tercetak untuk dibandingkan.
#
# Kalau angka di bawah COCOK dengan acuan lokal, inputnya identik dan selisih hasil apa
# pun lahir di training (torch/cuDNN). Kalau TIDAK cocok, inputnya sendiri yang beda dan
# tidak ada gunanya menyalahkan training.
import scipy as _sp
print("\nversi: numpy %s | scipy %s | torch %s"%(np.__version__, _sp.__version__, torch.__version__))
print("acuan lokal: numpy 1.26.4 | scipy 1.17.1 | torch 2.6.0+cu124")
_ACUAN = {"Wtr": 1.964203e+07, "Str": 1.907536e+05,
          "Wte": 4.208731e+06, "Ste": 4.244618e+04, "A_BASE": 4.658112e+05}
print("\nSIDIK JARI INPUT            terhitung        acuan lokal   selisih")
_ada_beda = False
_AB = ["Cd_2","Cv_2","Ca_4","logdist","loghypo","logE","snr_db"]   # = A_BASE, didefinisikan
for _nm, _a in [("Wtr",Wtr), ("Str",Str), ("Wte",Wte), ("Ste",Ste),   # di sel berikutnya
                ("A_BASE", Ftr[_AB].values)]:
    _v = float(np.nansum(np.abs(np.asarray(_a, dtype=np.float64))))
    _rel = (_v-_ACUAN[_nm])/_ACUAN[_nm]
    if abs(_rel) > 1e-5: _ada_beda = True   # 1e-5: longgar thd urutan penjumlahan float
    print("  %-8s %18.6e %18.6e  %+9.2e"%(_nm, _v, _ACUAN[_nm], _rel))
print("  nonfinite di Ftr:", int((~np.isfinite(Ftr.values.astype(np.float64))).sum()),
      " (acuan lokal: 18, semuanya dari `kurt`)")
print("  -> " + ("INPUT BEDA dari lokal. Selisih hasil lahir di sini, bukan di training."
                 if _ada_beda else
                 "input identik dengan lokal. Selisih hasil apa pun lahir di training."))


In [ ]:
import torch.nn as nn
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
A_BASE=["Cd_2","Cv_2","Ca_4","logdist","loghypo","logE","snr_db"]

class Blk(nn.Module):
    def __init__(s, ci, co, k=7, pool=4):
        super().__init__()
        s.c1=nn.Conv1d(ci,co,k,padding=k//2); s.b1=nn.BatchNorm1d(co)
        s.c2=nn.Conv1d(co,co,k,padding=k//2); s.b2=nn.BatchNorm1d(co)
        s.se=nn.Sequential(nn.AdaptiveAvgPool1d(1),nn.Conv1d(co,max(4,co//8),1),nn.SiLU(),
                           nn.Conv1d(max(4,co//8),co,1),nn.Sigmoid())
        s.sk=nn.Conv1d(ci,co,1) if ci!=co else nn.Identity()
        s.act=nn.SiLU(); s.pool=nn.MaxPool1d(pool)
    def forward(s,x):
        h=s.act(s.b1(s.c1(x))); h=s.b2(s.c2(h)); h=h*s.se(h)
        return s.pool(s.act(h+s.sk(x)))

class Net3(nn.Module):
    """CNN v3 notebook_8 (LB 0,1849). Acuan, bukan yang diuji di sini."""
    def __init__(s, n_scalar, w=64):
        super().__init__()
        s.body=nn.Sequential(Blk(9,w),Blk(w,w*2),Blk(w*2,w*3),Blk(w*3,w*4,k=5,pool=2),
                             Blk(w*4,w*4,k=3,pool=2))
        s.trunk=nn.Sequential(nn.Linear(w*8+n_scalar,256), nn.SiLU(), nn.Dropout(0.25),
                              nn.Linear(256,96), nn.SiLU())
        s.mag=nn.Linear(96,1); s.dep=nn.Linear(96,1)
    def forward(s,x,sc_):
        h=s.body(x); h=torch.cat([h.mean(-1),h.amax(-1),sc_],dim=1); h=s.trunk(h)
        return s.mag(h).squeeze(-1), s.dep(h).squeeze(-1)

def run(Wtr,Str,Wte,Ste, sw, seeds=(0,1,2), epochs=80, bs=128, lr=2.5e-3,
        avg_last=10, resid=True, arch=None):
    """Loop training CNN v3 persis seperti notebook_8.

    Empat hal yang membuat v3 menang: (1) kepala kedua memprediksi log10(depth), dihitung
    HANYA untuk record depth-desimal (di sana depth adalah inversi sungguhan, bukan default
    katalog); (2) augmentasi derau FISIK -- tempel jendela pre-P record lain, jadi derau
    stasiun sungguhan bukan Gaussian sintetis; (3) blok Squeeze-Excitation; (4) jadwal
    panjang + rata-rata bobot 10 epoch terakhir."""
    Xa=Ftr[A_BASE].values.astype(np.float64); Xa_te=Fte[A_BASE].values.astype(np.float64)
    dep_t=np.log10(d_tr+1).astype(np.float32); dep_ok=(~rnd_tr).astype(np.float32)
    noise=Wtr[:,:,:200].copy()
    oof=np.zeros(len(y)); pte=np.zeros(len(Wte))
    Xte_t=torch.tensor(Wte).to(DEV); Ste_t=torch.tensor(Ste).to(DEV)
    nf=0; t0=time.time()
    for seed in seeds:
        torch.manual_seed(seed); np.random.seed(seed)
        for ti,vi in GroupKFold(5).split(Wtr,y,ev):
            if resid:
                rg=Ridge(alpha=1.0).fit(Xa[ti],y[ti],sample_weight=sw[ti])
                tgt=(y-rg.predict(Xa)).astype(np.float32)
                off_v=rg.predict(Xa[vi]); off_t=rg.predict(Xa_te)
            else:
                tgt=y.astype(np.float32); off_v=0; off_t=0
            net=(arch or Net3)(Str.shape[1]).to(DEV)
            opt=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=2e-2)
            sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,
                    total_steps=epochs*max(1,len(ti)//bs),pct_start=0.2)
            Xt=torch.tensor(Wtr[ti]); St=torch.tensor(Str[ti]); Nz=torch.tensor(noise[ti])
            tt=torch.tensor(tgt[ti]); wt=torch.tensor(sw[ti],dtype=torch.float32)
            dt=torch.tensor(dep_t[ti]); dk=torch.tensor(dep_ok[ti])
            Xv=torch.tensor(Wtr[vi]).to(DEV); Sv=torch.tensor(Str[vi]).to(DEV)
            av=np.zeros(len(vi)); at=np.zeros(len(Wte))
            for ep in range(epochs):
                net.train(); perm=torch.randperm(len(ti))
                for i in range(0,len(ti)-bs+1,bs):
                    b=perm[i:i+bs]; xb=Xt[b].clone(); sb=St[b].to(DEV)
                    th=torch.rand(len(b))*2*np.pi; cs,sn=torch.cos(th),torch.sin(th)
                    for k in (0,3,6):
                        e_,n_=xb[:,k].clone(),xb[:,k+1].clone()
                        xb[:,k]=cs[:,None]*e_-sn[:,None]*n_; xb[:,k+1]=sn[:,None]*e_+cs[:,None]*n_
                    if torch.rand(1).item()<0.5:
                        for k in (2,5,8): xb[:,k]*=-1
                    if torch.rand(1).item()<0.5:
                        j=torch.randint(0,len(ti),(len(b),))
                        amp=(torch.rand(len(b),1,1)*0.6)
                        xb=xb+amp*Nz[j].repeat(1,1,5)[:,:,:xb.shape[2]]
                    xb=xb.to(DEV)
                    pm,pdep=net(xb,sb)
                    lm=(wt[b].to(DEV)*(pm-tt[b].to(DEV)).abs()).sum()/wt[b].sum().to(DEV)
                    dkb=dk[b].to(DEV)
                    ld=((pdep-dt[b].to(DEV)).abs()*dkb).sum()/(dkb.sum()+1e-6)
                    opt.zero_grad(); (lm+0.15*ld).backward(); opt.step(); sch.step()
                if ep>=epochs-avg_last:
                    net.eval()
                    with torch.no_grad():
                        av+=net(Xv,Sv)[0].cpu().numpy()
                        at+=np.concatenate([net(Xte_t[i:i+512],Ste_t[i:i+512])[0].cpu().numpy()
                                            for i in range(0,len(Wte),512)])
            oof[vi]+=(av/avg_last+off_v)/len(seeds)
            pte+=(at/avg_last+off_t)/(5*len(seeds))
            nf+=1
            print("    fold %2d/%d selesai  (%.0f menit terpakai, ~%.0f menit tersisa)"
                  %(nf,5*len(seeds),(time.time()-t0)/60,
                    (time.time()-t0)/nf*(5*len(seeds)-nf)/60), flush=True)
    return oof,pte


In [ ]:
REF = np.load(REFP, allow_pickle=True) if REFP else None
CAND = [str(x) for x in REF["__cand__"]] if REF is not None else []

def ref_stack():
    O=np.stack([REF[f"{n}__oof"].astype(np.float64) for n in CAND])
    E=np.stack([REF[f"{n}__pte"].astype(np.float64) for n in CAND])
    return O,E

# Validasi ref SEKARANG, bukan berjam-jam kemudian setelah training selesai.
# Kalau file-nya benar, blend 8 kandidat itu harus mereproduksi lb_real notebook_8.
if REF is not None:
    _o = post_tr(REF["__w__"] @ ref_stack()[0]); _v = sc(_o)
    print("ref termuat: %d kandidat -> lb_real %.4f  (nb8 seharusnya 0,1762)"%(len(CAND),_v))
    print("   " + ("COCOK -- verdikt bisa dipercaya." if abs(_v-0.1762) < 0.0005 else
                   "TIDAK COCOK -- file ref salah/rusak, verdikt tidak bisa dipercaya."))
    del _o, _v

def blend_search(names, Otr, Ote, ref_oof, nsamp=8000, alpha=0.6, seed=0):
    """Protokol nb6/nb8: cari bobot di separuh event, ukur di separuh LAIN, 12 belahan;
    bobot final = rata-rata lintas belahan (bukan argmin di data penuh).

    Ini perlu. Waktu MLP divergen (OOF 0,778) ikut jadi kandidat di nb5, pencarian bobot
    tetap memberinya 0,36 dan 'memperbaiki' skor. Jaga kandidat tetap sedikit."""
    rng=np.random.RandomState(seed)
    W=np.vstack([rng.dirichlet(np.ones(len(names))*alpha,size=nsamp),np.eye(len(names))])
    Q=np.stack([post_tr(x@Otr) for x in W])
    uev=np.unique(ev); gains=[];opt=[];ws=[];dgains=[]
    for rep in range(6):
        r=np.random.RandomState(100+rep); perm=r.permutation(uev)
        h1=np.isin(ev,perm[:len(perm)//2])
        for keep in (h1,~h1):
            ss=Scorer(EVR,keep); sh=Scorer(EVR,~keep); shd=SideScorer(True,~keep)
            v=np.array([ss(q) for q in Q]); i=v.argmin()
            gains.append(sh(Q[i])-sh(ref_oof)); opt.append(sh(Q[i])-v[i]); ws.append(W[i])
            dgains.append(shd(Q[i])-shd(ref_oof))     # DIAGNOSTIK 4: gain di sisi desimal
    wm=np.mean(ws,0); wm/=wm.sum()
    o=post_tr(wm@Otr); p=post_te(wm@Ote); om=float(np.mean(opt))/np.sqrt(2)
    return dict(w=wm, oof=o, pte=p, lb_real=float(sc(o)), jujur=float(sc(o))+om,
                optimisme=om, gain=float(np.mean(gains)), menang=int(sum(g<0 for g in gains)),
                bias=bias_rnd(o), dec=part_real(o,True), rnd=part_real(o,False),
                dec_gain=float(np.mean(dgains)),
                dec_menang=int(sum(g<0 for g in dgains)),
                mean_rnd_te=float(p[MT].mean()), sd_rnd_te=float(p[rnd_te].std()))

TARGET = [("peringkat 1 (Kingpul)", 0.1837), ("0,1800", 0.1800),
          ("0,1750", 0.1750), ("0,1700", 0.1700), ("0,1600", 0.1600)]

def verdikt(res, nama):
    """Cetak diagnostik transfer + verdikt target. Tiga diagnostik wajib sebelum submit."""
    est = lb_hat(res["lb_real"])
    print("="*70); print("VERDIKT -- %s"%nama); print("="*70)
    print("  lb_real        %.4f   (notebook_8 = 0,1762)"%res["lb_real"])
    print("  lb_real jujur  %.4f   (+optimisme seleksi %.4f)"%(res["jujur"],res["optimisme"]))
    print("  gain vs nb8    %+.4f  (menang %d/12 belahan)"%(res["gain"],res["menang"]))
    print("  dec %.4f  rnd %.4f   (nb8: 0,1437 / 0,2074)"%(res["dec"],res["rnd"]))
    print()
    print("  --- diagnostik transfer (SEMUA wajib aman sebelum submit) ---")
    ok1 = res["bias"] <= 0.05
    ok2 = abs(res["mean_rnd_te"]-5.272) <= 0.03
    ok3 = res["sd_rnd_te"] <= 0.42
    ok4 = res["lb_real"] < LB_JUARA
    print("  bias_rnd            %+.3f   %s  (tolak kalau > 0,05; nb8 +0,031)"
          %(res["bias"], "AMAN " if ok1 else "BAHAYA"))
    print("  mean test bulat<100 %.3f    %s  (optimum ~5,272; nb8 5,259)"
          %(res["mean_rnd_te"], "AMAN " if ok2 else "AWAS  "))
    print("  sd test sisi bulat  %.3f    %s  (naik = LB memburuk; nb8 0,396)"
          %(res["sd_rnd_te"], "AMAN " if ok3 else "AWAS  "))
    print("  lb_real vs juara    %.4f   %s  (harus < %.4f = notebook_8)"
          %(res["lb_real"], "AMAN " if ok4 else "BAHAYA", LB_JUARA))
    print()
    print("  --- informasi saja, BUKAN kriteria seleksi ---")
    print("  gain sisi desimal   %+.4f  (menang %d/12)"
          %(res.get("dec_gain",float("nan")), res.get("dec_menang",-1)))
    print("  Jangan pakai angka ini untuk menerima/menolak model per-record. Aturan lama")
    print("  'gain sisi desimal harus negatif' DICABUT: notebook_10 dibangun untuk")
    print("  memaksimalkannya dan LB-nya rugi 0,0022 (0,1849 -> 0,1871). Sisi desimal")
    print("  sudah menyentuh oracle (0,1437 vs ~0,1488); mengoptimalkannya cuma")
    print("  mengorbankan sisi bulat, yaitu 51% metrik dan seluruh headroom tersisa.")
    print()
    print("  ESTIMASI LB = lb_real + 0,0096 = %.4f   (sd 0,0036 -- TIDAK STABIL)"%est)
    print("  --- verdikt target ---")
    for nm,t in TARGET:
        d=est-t
        print("    %-22s %.4f   %s (selisih %+.4f)"
              %(nm,t,"TERCAPAI" if d<=0 else "belum   ",d))
    print("="*70)
    if not ok4:
        print("BAHAYA -- lb_real blend ini TIDAK mengalahkan notebook_8 (0,1762).")
        print("JANGAN disubmit. `notebook_10` melanggar aturan ini atas dasar teori dan")
        print("rugi 0,0022. Satu kegagalan LB bukan dasar cukup untuk membatalkan metrik")
        print("peringkat yang benar 4 dari 5 kali.")
    if not (ok1 and ok2 and ok3):
        print("PERINGATAN: diagnostik transfer tidak aman. `lb_real` mengukur data TRAIN;")
        print("`bias_rnd` dan mean subset mengukur JARAK KE TEST. Kalau bertabrakan, yang")
        print("kedua menang -- notebook_2 dan notebook_5 dua-duanya CV-nya membaik lalu")
        print("LB-nya memburuk karena aturan ini dilanggar.")
    return est

HASIL = []          # file yang dihasilkan notebook ini, untuk dibundel di akhir

def catat(p):
    p = str(p)
    if p not in HASIL: HASIL.append(p)
    return p

def simpan(nama, oof, pte, res=None):
    catat(OUT/f"{nama}.npz"); np.savez(OUT/f"{nama}.npz", oof=oof, pte=pte)
    print("tersimpan:", OUT/f"{nama}.npz")
    if res is not None:
        sub=pd.DataFrame({"sample_id":test.sample_id,"magnitude":np.round(res["pte"],3)})
        sub.to_csv(catat(OUT/f"{nama}_submission.csv"), index=False)
        json.dump({k:(v if isinstance(v,str) else np.asarray(v).tolist() if not np.isscalar(v) else float(v))
                   for k,v in res.items() if k not in ("oof","pte")},
                  open(catat(OUT/f"{nama}_score.json"),"w"), indent=2)
        print("tersimpan:", OUT/f"{nama}_submission.csv")

def bundel(nama_zip):
    """Bungkus hasil notebook INI jadi satu zip supaya gampang diunduh dari Kaggle.

    Hanya file yang dicatat lewat `catat()` yang ikut -- waveform dan input CNN yang
    berukuran ratusan MB sengaja ditinggal, karena yang dibutuhkan di lokal cuma .npz
    prediksi, .csv submission, dan .json skor."""
    z = OUT/f"{nama_zip}.zip"
    files = [f for f in HASIL if os.path.exists(f)]
    if not files:
        print("tidak ada file untuk dibundel."); return None
    with zipfile.ZipFile(z,"w",zipfile.ZIP_DEFLATED) as zf:
        for f in files: zf.write(f, arcname=os.path.basename(f))
    tot = sum(os.path.getsize(f) for f in files)/1024
    print("\n" + "="*70)
    print("BUNDEL SIAP DIUNDUH: %s   (%d file, %.0f KB)"%(z, len(files), tot))
    for f in files: print("   -", os.path.basename(f))
    print("="*70)
    print("Unduh zip itu dari panel Output di kanan, lalu ekstrak .npz-nya ke")
    print("`Case 1/cache/` supaya bisa dipakai blend berikutnya di lokal.")
    return z


---
## 1. Pseudo-label + bobot + belahan test

Label-semu dihitung dari blend notebook_8 (`nb8_ref.npz`), sudah lewat `post_te`. Bobot sampelnya dibangun meniru konstruksi train (`w(M)/cnt`), dengan grup (depth, magnitudo dibulatkan ke 0,1) — magnitudo katalog memang diskrit.

Belahan test dibuat menurut `depth_km`, **bukan** acak per record: record satu gempa berbagi nilai depth, dan kalau terpisah antar-belahan, model yang melihat separuh gempa bisa mengenali separuh lainnya lewat grup depth.

In [ ]:
# ---------------------------------------------------------------------------------
# Pseudo-label: prediksi test dari blend notebook_8, sudah lewat post_te.
#
# Ini SATU-SATUNYA sumber label untuk record test. Kualitasnya = kualitas nb8
# (lb_real 0,1762 / LB 0,1849), jadi PL tidak bisa mengajarkan sesuatu yang nb8 tidak
# tahu -- yang ia ubah adalah KOMPOSISI data training, bukan isi pengetahuannya.
# ---------------------------------------------------------------------------------
assert REF is not None, "nb8_ref.npz wajib ada: pseudo-label dihitung dari blend nb8."
_Oref, _Eref = ref_stack()
PL = post_te(REF["__w__"] @ _Eref)
print("pseudo-label test: mean %.3f  sd %.3f  (train y: mean %.3f sd %.3f)"
      %(PL.mean(), PL.std(), y.mean(), y.std()))
print("proporsi depth-bulat (per record): train %.1f%%  test %.1f%%  -> gabungan %.1f%%"
      %(100*rnd_tr.mean(), 100*rnd_te.mean(),
        100*(rnd_tr.sum()+rnd_te.sum())/(len(y)+len(PL))))

# Bobot sampel untuk record test, dibangun MENIRU konstruksi train: w(M)/cnt, dengan
# cnt = ukuran grup (depth, magnitudo). Magnitudo test tidak diketahui, jadi dipakai
# pseudo-label yang dibulatkan ke grid 0,1 -- magnitudo katalog memang diskrit.
_plb = np.round(PL, 1)
_gte = pd.DataFrame({"d": d_te, "m": _plb}).groupby(["d", "m"]).ngroup().values
_cte = pd.Series(_gte).map(pd.Series(_gte).value_counts()).values
SW_PL = mag_weight(PL) / _cte
print("bobot PL: mean %.4f vs bobot train mean %.4f  (rasio %.2f)"
      %(SW_PL.mean(), SW.mean(), SW_PL.mean()/SW.mean()))

# Yang benar-benar dilihat loss adalah proporsi BERBOBOT, bukan per record -- loss-nya
# weighted L1. Ini angka yang menentukan seberapa besar mekanisme "geser komposisi"
# sebenarnya bekerja; kalau kecil, gain apa pun yang muncul bukan dari mekanisme ini.
_wr_tr = SW[rnd_tr].sum()/SW.sum()
_wr_all = (SW[rnd_tr].sum()+SW_PL[rnd_te].sum())/(SW.sum()+SW_PL.sum())
print("proporsi depth-bulat (BERBOBOT, yang dilihat loss): train %.1f%% -> +PL %.1f%%"
      %(100*_wr_tr, 100*_wr_all))
print("  porsi bobot total yang dipegang record PL: %.1f%%"
      %(100*SW_PL.sum()/(SW.sum()+SW_PL.sum())))

# Belahan test untuk mencegah kebocoran label-semu ke dirinya sendiri.
#
# Dibelah menurut depth_km supaya record dari satu gempa tidak terpisah -- kalau terpisah,
# model yang melihat setengah gempa akan "mengenali" setengah lainnya lewat grup depth.
#
# TAPI membelah nilai depth 50:50 menghasilkan 1633 vs 667 record, karena depth = 10,0
# sendirian memuat ~745 record. Itu cacat: kedua model jadi punya paparan label-semu yang
# sangat berbeda, sehingga separuh prediksi test dihasilkan model ber-PL kuat dan separuh
# lagi model ber-PL lemah. Jadi nilai depth dialokasikan GREEDY menurut jumlah record:
# yang terbesar duluan, selalu ke belahan yang saat itu lebih kosong.
_cnt_d = pd.Series(d_te).value_counts()
_half = {}; _tot = [0, 0]
for _d, _n in _cnt_d.items():
    _s = 0 if _tot[0] <= _tot[1] else 1
    _half[_d] = _s; _tot[_s] += _n
PL_HALF = pd.Series(d_te).map(_half).values.astype(int)
print("belahan test: %d vs %d record  (%d vs %d nilai depth)"
      %((PL_HALF==0).sum(), (PL_HALF==1).sum(),
        (np.array(list(_half.values()))==0).sum(), (np.array(list(_half.values()))==1).sum()))
assert pd.DataFrame({"d": d_te, "h": PL_HALF}).groupby("d").h.nunique().max() == 1, \
    "belahan memecah grup depth -- record satu gempa tidak boleh terpisah"


def run_pl(Wtr, Str, Wte, Ste, sw, pl=None, pl_w=1.0, seeds=(0,1,2), epochs=80,
           bs=128, lr=2.5e-3, avg_last=10, tag=""):
    """`Net3` + loop training notebook_8 PERSIS. Satu-satunya perubahan: kalau `pl`
    diberikan, record test berlabel-semu ikut masuk ke himpunan training tiap fold.

    `pl=None`   -> arm A, kontrol. Harus mereproduksi cnn3 (post: 0,1898).
    `pl=PL`     -> arm B. Tiap fold melatih DUA model bersilang atas belahan test,
                   supaya tidak ada record test yang diprediksi model yang melihat
                   label-semunya sendiri.

    Anchor ridge SELALU difit di record train saja. Kalau anchor ikut berubah, gain/rugi
    tidak bisa dipisahkan antara "CNN belajar lain" dan "baseline linearnya bergeser" --
    pelajaran dari run B notebook 09."""
    Xa = Ftr[A_BASE].values.astype(np.float64); Xa_te = Fte[A_BASE].values.astype(np.float64)
    dep_t = np.log10(d_tr+1).astype(np.float32); dep_ok = (~rnd_tr).astype(np.float32)
    dep_t_te = np.log10(d_te+1).astype(np.float32); dep_ok_te = (~rnd_te).astype(np.float32)
    oof = np.zeros(len(y)); pte = np.zeros(len(Wte))
    Xte_t = torch.tensor(Wte).to(DEV); Ste_t = torch.tensor(Ste).to(DEV)
    # kalau pakai PL, tiap fold melatih 2 model -> pembagi prediksi OOF ikut 2x
    n_arm = 1 if pl is None else 2
    nf = 0; ntot = 5*len(seeds)*n_arm; t0 = time.time()

    for seed in seeds:
        torch.manual_seed(seed); np.random.seed(seed)
        for ti, vi in GroupKFold(5).split(Wtr, y, ev):
            rg = Ridge(alpha=1.0).fit(Xa[ti], y[ti], sample_weight=sw[ti])
            tgt = (y - rg.predict(Xa)).astype(np.float32)
            off_v = rg.predict(Xa[vi]); off_t = rg.predict(Xa_te)
            tgt_te = (pl - off_t).astype(np.float32) if pl is not None else None

            for arm in range(n_arm):
                if pl is None:
                    idx_te = np.zeros(0, dtype=int)       # tidak ada record test di training
                    pred_te = np.arange(len(Wte))          # prediksi seluruh test
                else:
                    idx_te = np.where(PL_HALF == arm)[0]   # yang IKUT dilatih
                    pred_te = np.where(PL_HALF != arm)[0]  # yang DIPREDIKSI (tidak dilatih)

                Xt = torch.tensor(np.concatenate([Wtr[ti], Wte[idx_te]]))
                St = torch.tensor(np.concatenate([Str[ti], Ste[idx_te]]))
                tt = torch.tensor(np.concatenate([tgt[ti], tgt_te[idx_te]])
                                  if pl is not None else tgt[ti])
                wt = torch.tensor(np.concatenate([sw[ti], SW_PL[idx_te]*pl_w])
                                  if pl is not None else sw[ti], dtype=torch.float32)
                dt = torch.tensor(np.concatenate([dep_t[ti], dep_t_te[idx_te]])
                                  if pl is not None else dep_t[ti])
                dk = torch.tensor(np.concatenate([dep_ok[ti], dep_ok_te[idx_te]])
                                  if pl is not None else dep_ok[ti])
                Nz = torch.tensor(np.ascontiguousarray(Xt.numpy()[:, :, :200]))
                n = len(Xt)

                net = Net3(Str.shape[1]).to(DEV)
                opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=2e-2)
                sch = torch.optim.lr_scheduler.OneCycleLR(
                    opt, max_lr=lr, total_steps=epochs*max(1, n//bs), pct_start=0.2)
                Xv = torch.tensor(Wtr[vi]).to(DEV); Sv = torch.tensor(Str[vi]).to(DEV)
                av = np.zeros(len(vi)); at = np.zeros(len(pred_te))

                for ep in range(epochs):
                    net.train(); perm = torch.randperm(n)
                    for i in range(0, n-bs+1, bs):
                        b = perm[i:i+bs]; xb = Xt[b].clone(); sb = St[b].to(DEV)
                        th = torch.rand(len(b))*2*np.pi; cs, sn = torch.cos(th), torch.sin(th)
                        for k in (0, 3, 6):
                            e_, n_ = xb[:, k].clone(), xb[:, k+1].clone()
                            xb[:, k] = cs[:, None]*e_ - sn[:, None]*n_
                            xb[:, k+1] = sn[:, None]*e_ + cs[:, None]*n_
                        if torch.rand(1).item() < 0.5:
                            for k in (2, 5, 8): xb[:, k] *= -1
                        if torch.rand(1).item() < 0.5:
                            j = torch.randint(0, n, (len(b),))
                            amp = (torch.rand(len(b), 1, 1)*0.6)
                            xb = xb + amp*Nz[j].repeat(1, 1, 5)[:, :, :xb.shape[2]]
                        xb = xb.to(DEV)
                        pm, pdep = net(xb, sb)
                        wb = wt[b].to(DEV)
                        lm = (wb*(pm - tt[b].to(DEV)).abs()).sum()/wb.sum()
                        dkb = dk[b].to(DEV)
                        ld = ((pdep - dt[b].to(DEV)).abs()*dkb).sum()/(dkb.sum()+1e-6)
                        opt.zero_grad(); (lm + 0.15*ld).backward(); opt.step(); sch.step()
                    if ep >= epochs - avg_last:
                        net.eval()
                        with torch.no_grad():
                            av += net(Xv, Sv)[0].cpu().numpy()
                            at += np.concatenate([
                                net(Xte_t[pred_te][i:i+512], Ste_t[pred_te][i:i+512])[0]
                                .cpu().numpy() for i in range(0, len(pred_te), 512)])

                oof[vi] += (av/avg_last + off_v)/(len(seeds)*n_arm)
                pte[pred_te] += (at/avg_last + off_t[pred_te])/(5*len(seeds))
                nf += 1
                print("    %sfold %2d/%d selesai  (%.0f menit terpakai, ~%.0f menit tersisa)"
                      % (tag, nf, ntot, (time.time()-t0)/60,
                         (time.time()-t0)/nf*(ntot-nf)/60), flush=True)
    return oof, pte


In [ ]:
def diagnosa(oof, pte, nama, ref_o=None, ref_p=None):
    """Diagnostik yang menentukan nasib kandidat PL, dicetak SEBELUM blend apa pun.

    Yang paling penting bukan skornya, melainkan baris KOLAPS TEST. Pseudo-labeling bisa
    membuat prediksi test runtuh ke arah nb8 sementara OOF tetap terlihat beragam; kalau
    itu terjadi, pencarian bobot blend membaca diversitas yang tidak ada di test."""
    po = post_tr(oof)
    print("\n--- %s ---" % nama)
    print("  lb_real %.4f  dec %.4f  rnd %.4f  bias_rnd %+.3f"
          % (lb_real(po), part_real(po, True), part_real(po, False), bias_rnd(po)))
    print("  (pembanding LOKAL, semua SETELAH post: cnn3 0,1898 | cnn3_avg 0,1888)")
    if ref_o is not None:
        c_o = float(np.corrcoef(oof, ref_o)[0, 1])
        c_p = float(np.corrcoef(pte, ref_p)[0, 1])
        print("\n  KOLAPS TEST -- korelasi terhadap kandidat KONTROL se-lingkungan:")
        print("    oof  %.4f" % c_o)
        print("    pte  %.4f" % c_p)
        print("    selisih pte-oof %+.4f" % (c_p - c_o))
        if c_p - c_o > 0.02:
            print("    BAHAYA: prediksi test jauh lebih mirip daripada OOF-nya.")
            print("    Label-semu bocor ke prediksi test. Verdikt blend di bawah TIDAK sah")
            print("    -- bobot dipilih dari diversitas OOF yang tidak ada di test.")
        else:
            print("    AMAN: diversitas OOF dan test sebanding, tidak ada kolaps.")
    e = np.abs(oof - y)
    print("\n  MAE per subset:  desimal %.4f | bulat-dangkal %.4f | bulat-dalam %.4f"
          % (e[~rnd_tr].mean(), e[MR].mean(), e[rnd_tr & (d_tr >= 150)].mean()))
    return po


---
## 2. Arm A — KONTROL: resep notebook_8 apa adanya

**Uji kewarasan lingkungan, bukan cuma baseline.** `lb_real` harus mendarat di **0,1898** — angka **SETELAH `post_tr`**. Jangan bandingkan dengan 0,2091; itu `lb_real` MENTAH, dan tertukarnya dua jenis angka itulah yang merusak verdikt notebook 09.

Arm ini juga jadi pembanding se-lingkungan untuk verdikt blend, sehingga gap Kaggle-vs-lokal 0,0049 tidak lagi terhitung sebagai kerugian ide.

In [ ]:
sw = make_sw(power=0.5)
t0 = time.time()
oofA, pteA = run_pl(Wtr, Str, Wte, Ste, sw, pl=None, tag="A ")
simpan("cnn14a", oofA, pteA)
print("\narm A selesai %.0f menit"%((time.time()-t0)/60))

poA = diagnosa(oofA, pteA, "arm A = KONTROL (resep cnn3)")

KONTROL_POST = 0.1898   # lb_real cnn3 lokal SETELAH post_tr (mentahnya 0,2091)
_gap = lb_real(poA) - KONTROL_POST
print("\nKONTROL: %.4f vs %.4f lokal  -> gap %+.4f"
      %(lb_real(poA), KONTROL_POST, _gap))
print("  " + ("COCOK -- lingkungan Kaggle mereproduksi run lokal."
              if abs(_gap) < 0.004 else
              "TIDAK COCOK. Verdikt blend tetap SAH karena arm B dibandingkan\n  terhadap arm A (se-lingkungan), tapi catat angkanya: ini gap lingkungan yang\n  belum dijelaskan. Bandingkan juga sidik jari input di sel Setup."))


---
## 3. Arm B — + pseudo-label

2.300 record test masuk ke training tiap fold dengan label dari blend nb8. Tiap fold melatih **dua** model bersilang atas belahan test, jadi tidak ada record test yang diprediksi model yang melihat label-semunya sendiri.

> **Baris yang paling menentukan bukan `lb_real`, melainkan KOLAPS TEST.** Kalau korelasi `pte` terhadap arm A melonjak jauh di atas korelasi `oof`-nya, label-semu bocor ke prediksi test dan verdikt blend tidak sah — berapa pun bagus skornya.

In [ ]:
t0 = time.time()
oofB, pteB = run_pl(Wtr, Str, Wte, Ste, sw, pl=PL, pl_w=1.0, tag="B ")
simpan("cnn14b", oofB, pteB)
print("\narm B selesai %.0f menit"%((time.time()-t0)/60))

poB = diagnosa(oofB, pteB, "arm B = + pseudo-label", ref_o=oofA, ref_p=pteA)

print("\n--- efek terisolasi PL (positif = membaik) ---")
print("  lb_real      %+.4f"%(lb_real(poA)-lb_real(poB)))
print("  sisi bulat   %+.4f   <- ke sinilah PL seharusnya bekerja"
      %(part_real(poA,False)-part_real(poB,False)))
print("  sisi desimal %+.4f"%(part_real(poA,True)-part_real(poB,True)))
print("  bias_rnd     %+.3f -> %+.3f  (ambang 0,05)"%(bias_rnd(poA), bias_rnd(poB)))
print("\nKalau gain-nya ADA tapi tidak menumpuk di sisi bulat, mekanisme yang")
print("diklaim (komposisi 32%% -> 51%%) BUKAN yang bekerja -- catat itu.")


---
## 4. Verdikt blend — protokol UJI 2, pembanding se-lingkungan

Dua uji dijalankan:

- **UJI 1** — `cnn14b` menggantikan `cnn3_avg` di blend nb8. Pembandingnya blend nb8 apa adanya, jadi gap lingkungan ikut terhitung sebagai kerugian ide. Dicetak supaya sebanding dengan angka batch sebelumnya.
- **UJI 2** — pembandingnya blend nb8 dengan slot `cnn3_avg` diisi **arm A**. Kandidat dan pembanding sama-sama dari Kaggle. **Ini yang menentukan.**

Ambang: gain holdout **< 0** DAN menang **≥ 7/12** DAN `bias_rnd` ≤ 0,05.

In [ ]:
def uji(nama, o, p, base_names, base_O, base_E, ref_oof):
    keep = [i for i, n in enumerate(base_names) if n != "cnn3_avg"]
    names = [base_names[i] for i in keep] + [nama]
    Otr = np.vstack([base_O[keep], o[None, :]])
    Ote = np.vstack([base_E[keep], p[None, :]])
    r = blend_search(names, Otr, Ote, ref_oof)
    r["_nama"] = nama; r["_names"] = names
    return r

Oref, Eref = ref_stack()
ref_nb8 = post_tr(REF["__w__"] @ Oref)
print("acuan notebook_8: lb_real=%.4f   [LB aktual 0,1849]\n"%sc(ref_nb8))

print("=== UJI 1 -- pembanding blend nb8 apa adanya ===")
r1 = uji("cnn14b", oofB, pteB, CAND, Oref, Eref, ref_nb8)
print("cnn14b (ganti)   lb_real %.4f  jujur %.4f  gain %+.5f  menang %2d/12  bias %+.3f"
      %(r1["lb_real"], r1["jujur"], r1["gain"], r1["menang"], r1["bias"]))

print("\n=== UJI 2 -- pembanding = blend nb8 dgn cnn3_avg DIGANTI arm A ===")
rA = uji("cnn14a", oofA, pteA, CAND, Oref, Eref, ref_nb8)
print("  pembanding internal (arm A di slot CNN): lb_real %.4f"%rA["lb_real"])
r2 = uji("cnn14b", oofB, pteB, CAND, Oref, Eref, rA["oof"])
print("cnn14b (vs armA) lb_real %.4f  jujur %.4f  gain %+.5f  menang %2d/12  bias %+.3f"
      %(r2["lb_real"], r2["jujur"], r2["gain"], r2["menang"], r2["bias"]))


---
## 5. Verdikt akhir

Verdikt dibaca dari **UJI 2**. Kalau gerbang KOLAPS TEST di sel 3 menyala, verdikt ini tidak sah berapa pun angkanya — itu diperiksa lebih dulu.

In [ ]:
_ck = float(np.corrcoef(pteB, pteA)[0,1]) - float(np.corrcoef(oofB, oofA)[0,1])
kolaps = _ck > 0.02
est = verdikt(r2, "cnn14b (vs arm A)")
lolos = (r2["gain"] < 0 and r2["menang"] >= 7 and r2["bias"] <= 0.05
         and not kolaps)
print()
if kolaps:
    print("DIBATALKAN oleh gerbang KOLAPS TEST (selisih korelasi %+.4f > 0,02)."%_ck)
    print("Label-semu bocor ke prediksi test: bobot blend dipilih dari diversitas")
    print("OOF yang tidak ada di test. Angka di atas TIDAK bisa dipakai.")
elif lolos:
    print("LOLOS PROTOKOL -- layak jadi kandidat submission (notebook_11).")
    print("Serap di lokal dengan tools/serap_nb12.py sebelum menulis submission.")
else:
    print("TIDAK LOLOS -- notebook_8 tetap submission terbaik.")
    print("Catat kegagalannya di sudah_run/README.md BESERTA ALASANNYA, dan")
    print("catat juga apakah gain-nya menumpuk di sisi bulat seperti yang")
    print("diklaim mekanismenya. Yang gagal tanpa catatan pasti dicoba ulang.")
print()
print("Ingat: hanya ADA SATU submission final, dan selisih ke peringkat 1 (0,0014)")
print("ada di dalam derau LB publik (sd ~0,004 untuk model berbeda). Kandidat yang")
print("cuma menang tipis BUKAN alasan cukup untuk menggeser notebook_8.")


---
## 6. Bungkus jadi satu zip

`cnn14a` (kontrol) **wajib** ikut diunduh, bukan cuma `cnn14b` — tanpa arm A, verdikt UJI 2 tidak bisa dihitung ulang di lokal.

In [ ]:
bundel("hasil_cnn14_pl")
